# 13 CFM-ID 4.0 小规模 Pilot 与运行条件确认

**目的**：在 ChemVista 继续锁定的前提下，从冻结的 Development 集合中确定性选择 150 个代表化合物，
使用 CFM-ID 4.0 分别生成 `[M+H]+` 与 `[M-H]-` 的 10/20/40 eV 模拟谱，并完成运行成功率、资源消耗、
失败结构及模拟谱—公共实测谱（sim-to-real）一致性审计。

本 Notebook 对应后续计划中的 **Stage B / 13**，不是最终的大规模 CFM 生成。其结论将决定 14 号 Notebook
是否可以冻结大规模候选结构及任务清单。

**严格边界**：

- 仅从 `development_compounds.csv` 选择结构；不允许 External Novel 进入 Pilot。
- 不读取 ChemVista 谱图文件，也不读取 ChemVista 盲测标签；只读取冻结拆分角色以执行排除检查。
- CFM 模拟谱只用于数据扩增/预训练研究，不作为新的独立实验验证。
- 固定 Docker 标签 `wishartlab/cfmid:4.2.6.0`，并记录实际镜像 digest；不使用漂移的 `latest` 标签。
- 先运行一个结构的正、负模式冒烟测试；确认无误后再打开完整 Pilot 开关。

CFM-ID 官方说明：每次 `cfm-predict` 输出 `energy0/1/2`，分别对应 10/20/40 eV。
官方 Docker/CLI 文档：<https://hub.docker.com/r/wishartlab/cfmid>；项目主页：<https://cfmid.wishartlab.com/>。

## 0. 第一次运行前只需要看这里

1. 在 Windows 中启动 **Docker Desktop**。
2. 第一次运行保持 `RUN_FULL_PILOT = False`，从头运行 Notebook：它会选出 150 个化合物并检查环境。
3. 若提示镜像不存在，把 `PULL_DOCKER_IMAGE` 改成 `True`，只重新运行环境检查单元格（镜像较大，下载需要时间）。
4. 冒烟测试的正、负模式都成功且能解析出三个能量层后，把 `RUN_FULL_PILOT` 改成 `True`，从“CFM 执行”开始向下运行。

Notebook 支持断点续跑：已有且可解析的输出不会重复计算。若确需重新计算失败任务，把
`RERUN_FAILED = True`；不要手工删除整个输出目录。

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import pickle
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)
sns.set_theme(style='whitegrid', context='notebook')

# -------------------- 用户配置区 --------------------
PILOT_N = 150
SEED = 20260905
CFM_IMAGE = 'wishartlab/cfmid:4.2.6.0'
PROBABILITY_THRESHOLD = 0.001
COLLISION_ENERGIES = {0: 10, 1: 20, 2: 40}

PULL_DOCKER_IMAGE = True  # 第一次若镜像不存在，再改 True
RUN_SMOKE_TEST = True       # 先各跑一个正、负模式任务
RUN_FULL_PILOT = True      # 冒烟测试成功后改 True
RERUN_FAILED = False        # 仅在明确要重试失败任务时改 True
COMMAND_TIMEOUT_SECONDS = 30 * 60
MAX_FULL_JOBS = None        # 调试时可设 10；正式 Pilot 必须恢复 None
# --------------------------------------------------

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((p for p in candidates if (p / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置 ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
SPLIT_DIR = BASE / 'split_v1'
AUDIT_DIR = BASE / 'spectrum_audit_v1'
OUTPUT_DIR = BASE / 'cfm_v1' / 'pilot_v1'
RAW_DIR = OUTPUT_DIR / 'raw'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for path in (OUTPUT_DIR, RAW_DIR / 'positive', RAW_DIR / 'negative', FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

PATHS = {
    'development': SPLIT_DIR / 'development_compounds.csv',
    'split_manifest': SPLIT_DIR / 'compound_split_manifest.csv',
    'public_manifest': AUDIT_DIR / 'public_relevant_spectrum_manifest.csv.gz',
    'public_stream': ROOT / '质谱数据' / 'EnvMolBench_Cleaned.pkl',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少输入文件：\n' + '\n'.join(missing)

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.executable)
print('RDKit:', rdBase.rdkitVersion)
print('运行开关:', {'pull': PULL_DOCKER_IMAGE, 'smoke': RUN_SMOKE_TEST, 'full': RUN_FULL_PILOT})

## 1. 读取冻结输入并执行无泄漏检查

`compound_split_manifest.csv` 在这里只读取三个角色/身份字段，用于确认 Pilot 身份不属于 External Novel；
不使用其中任何 PBT 标签。公共谱图只选择 20 ppm 主分析合格、Development、去除完全重复后的谱图。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

development = pd.read_csv(PATHS['development'])
split_roles = pd.read_csv(
    PATHS['split_manifest'],
    usecols=['inchikey_2d', 'supervised_compound_role', 'chemvista_evaluation_role'],
)
public = pd.read_csv(PATHS['public_manifest'])

assert development['inchikey_2d'].is_unique
assert development['supervised_compound_role'].astype(str).eq('development').all()
external_novel_keys = set(
    split_roles.loc[
        split_roles['supervised_compound_role'].astype(str).eq('external_novel'),
        'inchikey_2d',
    ].astype(str)
)
assert not (set(development['inchikey_2d'].astype(str)) & external_novel_keys)

required_public = {
    'spectrum_id', 'inchikey_2d', 'stream_offset', 'precursor_mz', 'ion_mode', 'adduct',
    'collision_energy_value', 'cleaned_peak_n', 'primary_train_eligible_20ppm',
    'exact_dedup_keep', 'supervised_compound_role', 'chemvista_evaluation_role',
}
assert not (required_public - set(public.columns)), sorted(required_public - set(public.columns))
public_eligible = public.loc[
    public['primary_train_eligible_20ppm'].fillna(False).astype(bool)
    & public['exact_dedup_keep'].fillna(False).astype(bool)
    & public['supervised_compound_role'].astype(str).eq('development')
].copy()
assert not public_eligible['supervised_compound_role'].astype(str).eq('external_novel').any()

print('Development compounds:', len(development))
print('External Novel identities excluded:', len(external_novel_keys))
print('Eligible public measured spectra:', len(public_eligible))
print(public_eligible['ion_mode'].value_counts().to_string())

## 2. 构建 150 个代表化合物的 Pilot

选择仅依赖 Development 数据。首先用 RDKit 校验结构并计算分子量、元素组成；随后按以下轴做确定性轮转抽样：

- 低/中/高分子量；
- 公共实测谱的正、负模式覆盖；
- P、B、T 五个主要端点的标签可用组合；
- CHNO、卤素、S/P 和其他元素类别；
- 冻结五折结构簇。

这不是为模型挑选“容易样本”，而是为了用较少任务尽早暴露 CFM 的运行边界。

In [ ]:
PRIMARY_LABELS = {
    'P': 'label__P_screen_not_ready_binary',
    'B': 'label__B_LOGBCF_REGRESSION',
    'EC50': 'label__T_EC50_MG_L_REGRESSION',
    'LC50': 'label__T_LC50_MG_L_REGRESSION',
    'NOEC': 'label__T_NOEC_MG_L_REGRESSION',
}

def molecule_descriptors(smiles):
    mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
    if mol is None:
        return pd.Series({'valid_smiles': False, 'canonical_smiles_cfm': '', 'exact_mw': np.nan,
                          'heavy_atoms': np.nan, 'elements': '', 'element_class': 'invalid'})
    canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
    elements = sorted({atom.GetSymbol() for atom in mol.GetAtoms()})
    element_set = set(elements)
    if element_set & {'F', 'Cl', 'Br', 'I'}:
        element_class = 'halogen'
    elif element_set & {'S', 'P'}:
        element_class = 'S_or_P'
    elif element_set <= {'C', 'H', 'N', 'O'}:
        element_class = 'CHNO'
    else:
        element_class = 'other_element'
    return pd.Series({
        'valid_smiles': True,
        'canonical_smiles_cfm': canonical,
        'exact_mw': float(Descriptors.ExactMolWt(mol)),
        'heavy_atoms': int(mol.GetNumHeavyAtoms()),
        'elements': ','.join(elements),
        'element_class': element_class,
    })

mode_table = (
    public_eligible.assign(mode=public_eligible['ion_mode'].astype(str).str.lower())
    .groupby('inchikey_2d')['mode'].agg(lambda x: set(x)).to_dict()
)

pool = development.copy()
pool = pd.concat([pool, pool['canonical_smiles'].apply(molecule_descriptors)], axis=1)
pool = pool.loc[pool['valid_smiles'] & pool['inchikey_2d'].astype(str).isin(mode_table)].copy()
pool['measured_mode_profile'] = pool['inchikey_2d'].map(
    lambda key: 'both' if {'positive', 'negative'} <= mode_table.get(key, set())
    else ('positive_only' if 'positive' in mode_table.get(key, set()) else 'negative_only')
)
pool['mw_band'] = pd.qcut(pool['exact_mw'], 3, labels=['low', 'mid', 'high'], duplicates='drop').astype(str)
for short, column in PRIMARY_LABELS.items():
    pool[f'has_{short}'] = pd.to_numeric(pool[column], errors='coerce').notna()
pool['endpoint_signature'] = pool.apply(
    lambda row: ''.join(short for short in PRIMARY_LABELS if bool(row[f'has_{short}'])) or 'none', axis=1
)
pool['n_primary_endpoints'] = pool[[f'has_{x}' for x in PRIMARY_LABELS]].sum(axis=1).astype(int)
pool['cv_fold'] = pd.to_numeric(pool['cv_fold'], errors='coerce').astype('Int64')
pool['selection_cell'] = (
    pool['mw_band'] + '|' + pool['measured_mode_profile'] + '|' + pool['element_class']
    + '|fold' + pool['cv_fold'].astype(str) + '|n' + pool['n_primary_endpoints'].astype(str)
)

rng = np.random.default_rng(SEED)
pool['_random_order'] = rng.random(len(pool))
pool['_within_cell_rank'] = pool.groupby('selection_cell')['_random_order'].rank(method='first')

# 先固定低/中/高分子量配额，避免复合分层名称的排序影响 MW 分布；
# 再在每个 MW 档内部对模式、元素、端点数和冻结折的复合单元做轮转抽样。
mw_levels = [level for level in ['low', 'mid', 'high'] if level in set(pool['mw_band'])]
assert len(mw_levels) == 3, f'无法形成三个 MW 档：{mw_levels}'
base_quota, remainder = divmod(PILOT_N, len(mw_levels))
selected_parts = []
for level_index, level in enumerate(mw_levels):
    quota = base_quota + int(level_index < remainder)
    band = pool.loc[pool['mw_band'].eq(level)].copy()
    cells = sorted(band['selection_cell'].unique())
    shuffled_cells = list(rng.permutation(cells))
    cell_order = {cell: order for order, cell in enumerate(shuffled_cells)}
    band['_cell_order'] = band['selection_cell'].map(cell_order)
    chosen = band.sort_values(
        ['_within_cell_rank', '_cell_order', '_random_order', 'inchikey_2d']
    ).head(quota)
    assert len(chosen) == quota, f'{level} MW 档合格化合物不足：需要 {quota}，仅有 {len(band)}'
    selected_parts.append(chosen)
pilot = pd.concat(selected_parts, ignore_index=True)
pilot = pilot.sort_values(['_random_order', 'inchikey_2d']).reset_index(drop=True)
pilot.insert(0, 'pilot_id', [f'CFMPILOT_{i:04d}' for i in range(1, len(pilot) + 1)])

assert len(pilot) == PILOT_N, f'合格候选只有 {len(pilot)}，少于 PILOT_N={PILOT_N}'
assert pilot['inchikey_2d'].is_unique
assert not (set(pilot['inchikey_2d'].astype(str)) & external_novel_keys)
assert pilot['supervised_compound_role'].astype(str).eq('development').all()
assert pilot['canonical_smiles_cfm'].str.len().gt(0).all()

pilot_columns = [
    'pilot_id', 'inchikey_2d', 'preferred_names', 'canonical_smiles_cfm', 'exact_mw',
    'heavy_atoms', 'elements', 'element_class', 'mw_band', 'measured_mode_profile',
    'endpoint_signature', 'n_primary_endpoints', 'cv_group', 'cv_fold',
    'chemvista_evaluation_role', *PRIMARY_LABELS.values(),
]
pilot[pilot_columns].to_csv(OUTPUT_DIR / 'cfm_pilot_compounds.csv', index=False, encoding='utf-8-sig')

print('Eligible structure pool:', len(pool))
print('Pilot compounds:', len(pilot))
display(pilot[pilot_columns[:14]].head(10))

In [ ]:
endpoint_counts = pd.DataFrame({
    'endpoint': list(PRIMARY_LABELS),
    'n_labeled_in_pilot': [int(pilot[f'has_{x}'].sum()) for x in PRIMARY_LABELS],
})
categorical_counts = pd.concat([
    pilot.groupby('mw_band', dropna=False).size().rename('n').reset_index().rename(columns={'mw_band': 'level'}).assign(axis='MW band'),
    pilot.groupby('measured_mode_profile', dropna=False).size().rename('n').reset_index().rename(columns={'measured_mode_profile': 'level'}).assign(axis='Measured modes'),
    pilot.groupby('element_class', dropna=False).size().rename('n').reset_index().rename(columns={'element_class': 'level'}).assign(axis='Element class'),
    pilot.groupby('cv_fold', dropna=False).size().rename('n').reset_index().rename(columns={'cv_fold': 'level'}).assign(axis='Frozen fold'),
], ignore_index=True)
endpoint_counts.to_csv(OUTPUT_DIR / 'cfm_pilot_endpoint_coverage.csv', index=False)
categorical_counts.to_csv(OUTPUT_DIR / 'cfm_pilot_selection_balance.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.histplot(data=pilot, x='exact_mw', hue='mw_band', bins=20, ax=axes[0], legend=True)
axes[0].set_title('Pilot molecular-weight coverage')
sns.countplot(data=pilot, x='measured_mode_profile', order=['both', 'positive_only', 'negative_only'], ax=axes[1])
axes[1].set_title('Measured public-spectrum modes')
axes[1].tick_params(axis='x', rotation=20)
sns.barplot(data=endpoint_counts, x='endpoint', y='n_labeled_in_pilot', ax=axes[2], color='#4472C4')
axes[2].set_title('Primary endpoint coverage')
axes[2].set_ylim(0, PILOT_N)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '13_pilot_selection_coverage.png', dpi=240, bbox_inches='tight')
plt.show()

display(endpoint_counts)
display(categorical_counts.pivot_table(index='level', columns='axis', values='n', fill_value=0))

## 3. Docker 与 CFM-ID 版本预检

此处不会在后台静默改变软件版本。若 `PULL_DOCKER_IMAGE=False` 且本地没有固定镜像，会给出明确提示并跳过 CFM；
将其改成 `True` 才会执行 `docker pull wishartlab/cfmid:4.2.6.0`。

Windows 路径含中文时，Docker Desktop 必须有权访问 E 盘。冒烟测试若出现 `Mounts denied` 或路径错误，先在
Docker Desktop 中允许该磁盘/目录共享，再继续。

In [ ]:
def run_capture(command, timeout=120):
    started = time.perf_counter()
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding='utf-8',
                                errors='replace', timeout=timeout, shell=False)
        return {
            'ok': result.returncode == 0,
            'returncode': result.returncode,
            'stdout': result.stdout.strip(),
            'stderr': result.stderr.strip(),
            'seconds': time.perf_counter() - started,
        }
    except Exception as exc:
        return {'ok': False, 'returncode': None, 'stdout': '', 'stderr': repr(exc),
                'seconds': time.perf_counter() - started}

docker_executable = shutil.which('docker')
docker_client = run_capture([docker_executable, '--version']) if docker_executable else {
    'ok': False, 'stderr': 'docker executable not found', 'stdout': '', 'returncode': None, 'seconds': 0.0
}
docker_server = run_capture([docker_executable, 'info', '--format', '{{json .ServerVersion}}']) if docker_executable else docker_client
image_check = run_capture([docker_executable, 'image', 'inspect', CFM_IMAGE]) if docker_executable and docker_server['ok'] else {
    'ok': False, 'stderr': 'Docker server is not ready', 'stdout': '', 'returncode': None, 'seconds': 0.0
}

if docker_executable and docker_server['ok'] and not image_check['ok'] and PULL_DOCKER_IMAGE:
    print('固定 CFM-ID 镜像不存在，开始下载（只在 PULL_DOCKER_IMAGE=True 时执行）...')
    pull_result = run_capture([docker_executable, 'pull', CFM_IMAGE], timeout=2 * 60 * 60)
    print(pull_result['stdout'][-2000:])
    if not pull_result['ok']:
        print(pull_result['stderr'][-2000:])
    image_check = run_capture([docker_executable, 'image', 'inspect', CFM_IMAGE])
else:
    pull_result = None

if image_check['ok']:
    digest_result = run_capture([
        docker_executable, 'image', 'inspect',
        '--format', '{{json .RepoDigests}}|{{.Id}}', CFM_IMAGE,
    ])
    CFM_IMAGE_DIGEST = digest_result['stdout'] if digest_result['ok'] else 'unavailable'
else:
    CFM_IMAGE_DIGEST = 'unavailable'

ENVIRONMENT_READY = bool(docker_executable and docker_client['ok'] and docker_server['ok'] and image_check['ok'])
environment_table = pd.DataFrame([
    {'check': 'docker executable', 'passed': bool(docker_executable), 'detail': str(docker_executable)},
    {'check': 'docker client', 'passed': docker_client['ok'], 'detail': (docker_client['stdout'] or docker_client['stderr'])[-500:]},
    {'check': 'docker server', 'passed': docker_server['ok'], 'detail': (docker_server['stdout'] or docker_server['stderr'])[-500:]},
    {'check': 'fixed CFM image', 'passed': image_check['ok'], 'detail': CFM_IMAGE},
    {'check': 'image digest recorded', 'passed': CFM_IMAGE_DIGEST != 'unavailable', 'detail': CFM_IMAGE_DIGEST[-500:]},
])
environment_table.to_csv(OUTPUT_DIR / 'cfm_environment_checks.csv', index=False, encoding='utf-8-sig')
display(environment_table)

if not docker_server['ok']:
    print('\n下一步：启动 Docker Desktop，等待状态变为 Running，然后重新运行本单元格。')
elif not image_check['ok']:
    print('\n下一步：把 PULL_DOCKER_IMAGE 改为 True，再重新运行本单元格。')
else:
    print('\nCFM-ID 固定镜像已就绪，可以执行冒烟测试。')

## 4. 冻结任务清单

每个化合物有两个 CFM 命令任务：正模式 `[M+H]+`、负模式 `[M-H]-`。每个命令一次输出三个能量层，
因而 150 个化合物对应 300 个命令和 900 张模拟谱。命令使用参数列表执行（`shell=False`），避免 SMILES
中括号、括号等字符被 PowerShell 误解释。

In [ ]:
MODE_CONFIG = {
    'positive': {
        'adduct': '[M+H]+',
        'param': '/trained_models_cfmid4.0/[M+H]+/param_output.log',
        'config': '/trained_models_cfmid4.0/[M+H]+/param_config.txt',
    },
    'negative': {
        'adduct': '[M-H]-',
        'param': '/trained_models_cfmid4.0/[M-H]-/param_output.log',
        'config': '/trained_models_cfmid4.0/[M-H]-/param_config.txt',
    },
}

def relative_container_path(path):
    return '/cfmid/public/' + Path(path).relative_to(OUTPUT_DIR).as_posix()

jobs = []
for row in pilot.itertuples(index=False):
    for mode, settings in MODE_CONFIG.items():
        output_path = RAW_DIR / mode / f'{row.pilot_id}_{row.inchikey_2d}.txt'
        jobs.append({
            'job_id': f'{row.pilot_id}_{mode}',
            'pilot_id': row.pilot_id,
            'inchikey_2d': row.inchikey_2d,
            'canonical_smiles_cfm': row.canonical_smiles_cfm,
            'exact_mw': row.exact_mw,
            'elements': row.elements,
            'mode': mode,
            'adduct': settings['adduct'],
            'model_param': settings['param'],
            'model_config': settings['config'],
            'output_path': str(output_path),
            'output_container_path': relative_container_path(output_path),
        })
jobs = pd.DataFrame(jobs)
assert len(jobs) == PILOT_N * 2
assert not jobs.duplicated(['inchikey_2d', 'mode']).any()
jobs.to_csv(OUTPUT_DIR / 'cfm_job_manifest.csv', index=False, encoding='utf-8-sig')

spectrum_tasks = jobs.loc[jobs.index.repeat(3)].copy().reset_index(drop=True)
spectrum_tasks['energy_index'] = np.tile([0, 1, 2], len(jobs))
spectrum_tasks['collision_energy_ev'] = spectrum_tasks['energy_index'].map(COLLISION_ENERGIES)
spectrum_tasks['spectrum_task_id'] = (
    spectrum_tasks['job_id'] + '_ce' + spectrum_tasks['collision_energy_ev'].astype(str)
)
assert len(spectrum_tasks) == PILOT_N * 2 * 3
spectrum_tasks.to_csv(OUTPUT_DIR / 'cfm_spectrum_task_manifest.csv', index=False, encoding='utf-8-sig')

print('Command jobs:', len(jobs))
print('Expected simulated spectra:', len(spectrum_tasks))
display(jobs.head(4))

## 5. CFM 执行：先冒烟测试，再完整 Pilot

官方 `cfm-predict` 调用格式为：

`cfm-predict <SMILES> 0.001 <param_file> <config_file> <annotate_fragments> <output_file>`

这里令 `annotate_fragments=0`，因为后续 Transformer 只需要 m/z 与强度。每完成一个任务就刷新状态表，
因而中断后可以继续。状态表会保留返回码、耗时和标准错误摘要。

In [ ]:
ENERGY_HEADER = re.compile(r'^energy\s*([012])\s*$', flags=re.I)
PEAK_LINE = re.compile(
    r'^\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)'
    r'\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)'
)

def parse_cfm_output(path):
    spectra = {0: [], 1: [], 2: []}
    current = None
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return spectra
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        header = ENERGY_HEADER.match(line.strip())
        if header:
            current = int(header.group(1))
            continue
        peak = PEAK_LINE.match(line)
        if current is not None and peak:
            mz, intensity = float(peak.group(1)), float(peak.group(2))
            if np.isfinite(mz) and np.isfinite(intensity) and mz > 0 and intensity > 0:
                spectra[current].append((mz, intensity))
    return spectra

def output_is_complete(path):
    parsed = parse_cfm_output(path)
    return all(len(parsed[index]) > 0 for index in COLLISION_ENERGIES)

def docker_command(row):
    mount = f'{OUTPUT_DIR.resolve()}:/cfmid/public'
    return [
        docker_executable, 'run', '--rm', '-v', mount, '-w', '/cfmid/public',
        CFM_IMAGE, 'cfm-predict', row.canonical_smiles_cfm,
        str(PROBABILITY_THRESHOLD), row.model_param, row.model_config,
        '0', row.output_container_path,
    ]

STATUS_PATH = OUTPUT_DIR / 'cfm_execution_status.csv'
previous_status = pd.read_csv(STATUS_PATH) if STATUS_PATH.exists() else pd.DataFrame()
previous_by_job = previous_status.set_index('job_id').to_dict('index') if len(previous_status) else {}

def execute_job(row, force=False):
    output_path = Path(row.output_path)
    old = previous_by_job.get(row.job_id, {})
    if output_is_complete(output_path) and not force:
        return {
            'job_id': row.job_id, 'mode': row.mode, 'inchikey_2d': row.inchikey_2d,
            'status': 'cached_complete', 'returncode': 0,
            'runtime_seconds': old.get('runtime_seconds', np.nan),
            'started_at_utc': old.get('started_at_utc', ''),
            'finished_at_utc': old.get('finished_at_utc', ''),
            'output_bytes': output_path.stat().st_size,
            'energy0_peak_n': len(parse_cfm_output(output_path)[0]),
            'energy1_peak_n': len(parse_cfm_output(output_path)[1]),
            'energy2_peak_n': len(parse_cfm_output(output_path)[2]),
            'stderr_tail': old.get('stderr_tail', ''),
            'command_json': json.dumps(docker_command(row), ensure_ascii=False),
        }
    if not ENVIRONMENT_READY:
        return {
            'job_id': row.job_id, 'mode': row.mode, 'inchikey_2d': row.inchikey_2d,
            'status': 'not_run_environment_not_ready', 'returncode': np.nan,
            'runtime_seconds': np.nan, 'started_at_utc': '', 'finished_at_utc': '',
            'output_bytes': output_path.stat().st_size if output_path.exists() else 0,
            'energy0_peak_n': 0, 'energy1_peak_n': 0, 'energy2_peak_n': 0,
            'stderr_tail': 'Docker/CFM image preflight did not pass',
            'command_json': json.dumps(docker_command(row), ensure_ascii=False) if docker_executable else '',
        }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    started_at = utc_now()
    result = run_capture(docker_command(row), timeout=COMMAND_TIMEOUT_SECONDS)
    finished_at = utc_now()
    parsed = parse_cfm_output(output_path)
    complete = result['ok'] and all(len(parsed[i]) > 0 for i in COLLISION_ENERGIES)
    return {
        'job_id': row.job_id, 'mode': row.mode, 'inchikey_2d': row.inchikey_2d,
        'status': 'success' if complete else ('timeout_or_exception' if result['returncode'] is None else 'failed_or_incomplete'),
        'returncode': result['returncode'], 'runtime_seconds': result['seconds'],
        'started_at_utc': started_at, 'finished_at_utc': finished_at,
        'output_bytes': output_path.stat().st_size if output_path.exists() else 0,
        'energy0_peak_n': len(parsed[0]), 'energy1_peak_n': len(parsed[1]), 'energy2_peak_n': len(parsed[2]),
        'stderr_tail': (result['stderr'] or result['stdout'])[-2000:],
        'command_json': json.dumps(docker_command(row), ensure_ascii=False),
    }

def save_status(rows):
    frame = pd.DataFrame(rows)
    temporary = STATUS_PATH.with_suffix('.csv.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig')
    os.replace(temporary, STATUS_PATH)
    return frame

In [ ]:
# 冒烟测试分别选有相应模式公共实测谱、且 MW 接近 250 Da 的结构，
# 防止用明显不适合某一离子模式的第一个样本误判运行环境。
smoke_candidates = jobs.merge(
    pilot[['pilot_id', 'measured_mode_profile']], on='pilot_id', how='left'
)
smoke_parts = []
for smoke_mode in ['positive', 'negative']:
    allowed_profiles = {'both', f'{smoke_mode}_only'}
    available = smoke_candidates.loc[
        smoke_candidates['mode'].eq(smoke_mode)
        & smoke_candidates['measured_mode_profile'].isin(allowed_profiles)
    ].copy()
    available['_mw_distance_250'] = (available['exact_mw'] - 250.0).abs()
    smoke_parts.append(available.sort_values(['_mw_distance_250', 'job_id']).head(1))
smoke_rows = pd.concat(smoke_parts, ignore_index=True)[jobs.columns]
smoke_results = []
if RUN_SMOKE_TEST:
    if not ENVIRONMENT_READY:
        print('冒烟测试未执行：请先通过 Docker/CFM 镜像预检。')
    else:
        for row in smoke_rows.itertuples(index=False):
            print('Smoke:', row.job_id, row.adduct)
            result = execute_job(row, force=RERUN_FAILED)
            smoke_results.append(result)
            print(result['status'], 'runtime_s=', result['runtime_seconds'],
                  'peaks=', [result[f'energy{i}_peak_n'] for i in range(3)])
else:
    print('RUN_SMOKE_TEST=False：跳过冒烟测试。')

smoke_status = pd.DataFrame(smoke_results)
if len(smoke_status):
    smoke_status.to_csv(OUTPUT_DIR / 'cfm_smoke_test_status.csv', index=False, encoding='utf-8-sig')
    SMOKE_PASSED = smoke_status['status'].isin(['success', 'cached_complete']).all() and len(smoke_status) == 2
else:
    SMOKE_PASSED = all(output_is_complete(path) for path in smoke_rows['output_path'])
print('SMOKE_PASSED:', SMOKE_PASSED)
if ENVIRONMENT_READY and not SMOKE_PASSED:
    print('停止在冒烟阶段：检查 cfm_smoke_test_status.csv 的 stderr_tail，不要打开完整 Pilot。')
elif SMOKE_PASSED and not RUN_FULL_PILOT:
    print('冒烟测试通过。现在可把 RUN_FULL_PILOT 改为 True，再从下一单元格运行。')

In [ ]:
# 完整 Pilot。默认关闭；只有环境和冒烟测试均通过才会执行。
status_rows = []
if RUN_FULL_PILOT:
    assert MAX_FULL_JOBS is None, '正式 Pilot 要求 MAX_FULL_JOBS=None；调试截断不能当作完成结果。'
    assert ENVIRONMENT_READY, 'Docker/CFM 固定镜像预检未通过。'
    assert SMOKE_PASSED, '正、负模式冒烟测试未全部通过。'
    for index, row in enumerate(tqdm(jobs.itertuples(index=False), total=len(jobs), desc='CFM Pilot')):
        old = previous_by_job.get(row.job_id, {})
        force = bool(RERUN_FAILED and old.get('status') not in {'success', 'cached_complete'})
        status_rows.append(execute_job(row, force=force))
        if (index + 1) % 5 == 0 or index + 1 == len(jobs):
            save_status(status_rows)
    execution_status = save_status(status_rows)
elif STATUS_PATH.exists():
    execution_status = pd.read_csv(STATUS_PATH)
    print('RUN_FULL_PILOT=False：未启动批量任务，读取已有状态表。')
else:
    execution_status = pd.DataFrame(columns=['job_id', 'mode', 'inchikey_2d', 'status'])
    print('RUN_FULL_PILOT=False：当前只完成任务清单与冒烟测试。')

if len(execution_status):
    display(execution_status.groupby(['mode', 'status'], dropna=False).size().rename('n').reset_index())

## 6. 解析三能量 CFM 输出

只有 `energy0/1/2` 均包含有效峰的命令才计为成功。解析后保存长表，一行对应一个模拟碎片峰；
后续 15 号和 Transformer Notebook 应从该标准化长表读取，而不是再次猜测 CFM 原始文本格式。

In [ ]:
parsed_rows = []
spectrum_summary_rows = []
for row in tqdm(jobs.itertuples(index=False), total=len(jobs), desc='Parse CFM outputs'):
    spectra = parse_cfm_output(row.output_path)
    for energy_index, collision_energy in COLLISION_ENERGIES.items():
        peaks = spectra[energy_index]
        if peaks:
            maximum = max(intensity for _, intensity in peaks)
        else:
            maximum = np.nan
        for peak_rank, (mz, intensity) in enumerate(sorted(peaks), start=1):
            parsed_rows.append({
                'spectrum_task_id': f'{row.job_id}_ce{collision_energy}',
                'job_id': row.job_id, 'pilot_id': row.pilot_id,
                'inchikey_2d': row.inchikey_2d, 'mode': row.mode, 'adduct': row.adduct,
                'collision_energy_ev': collision_energy, 'energy_index': energy_index,
                'peak_rank_mz': peak_rank, 'fragment_mz': mz,
                'intensity_raw': intensity,
                'relative_intensity': intensity / maximum if np.isfinite(maximum) and maximum > 0 else np.nan,
            })
        spectrum_summary_rows.append({
            'spectrum_task_id': f'{row.job_id}_ce{collision_energy}',
            'job_id': row.job_id, 'pilot_id': row.pilot_id,
            'inchikey_2d': row.inchikey_2d, 'mode': row.mode, 'adduct': row.adduct,
            'collision_energy_ev': collision_energy, 'energy_index': energy_index,
            'peak_n': len(peaks), 'parse_success': len(peaks) > 0,
        })

parsed_peaks = pd.DataFrame(parsed_rows)
simulated_spectra = pd.DataFrame(spectrum_summary_rows)
simulated_spectra.to_csv(OUTPUT_DIR / 'cfm_simulated_spectrum_index.csv', index=False, encoding='utf-8-sig')
if len(parsed_peaks):
    parsed_peaks.to_csv(OUTPUT_DIR / 'cfm_parsed_spectra.csv.gz', index=False, compression='gzip')
print('Parsed spectra:', int(simulated_spectra['parse_success'].sum()), '/', len(simulated_spectra))
print('Parsed peaks:', len(parsed_peaks))

## 7. sim-to-real 诊断：与公共实测谱配对

为避免把不同离子形式混在一起，只比较同一二维 InChIKey、同一模式、且实测加合物恰为 `[M+H]+` 或 `[M-H]-`
的公共 Development 谱。每张模拟谱选择碰撞能量最接近的实测谱，要求绝对差不超过 10 eV；没有可比谱的任务不填零，
而是明确标记为无配对覆盖。

相似度为 0.02 Da 容差下、平方根强度变换后的贪心 cosine。它是生成质量诊断，不是 PBT 模型性能指标。

In [ ]:
MAX_MZ = 2000.0
MZ_TOLERANCE_DA = 0.02
MAX_CE_DIFFERENCE_EV = 10.0

def extract_observed_peaks(record):
    tensor = np.asarray(record['tensor'], dtype=np.float64)
    padding = np.asarray(record['padding_mask'], dtype=bool)
    valid = ~padding
    valid[0] = False
    mz = tensor[valid, 0] * MAX_MZ
    intensity = tensor[valid, 3]
    keep = np.isfinite(mz) & np.isfinite(intensity) & (mz > 0) & (intensity > 0)
    peaks = np.column_stack([mz[keep], intensity[keep]])
    return peaks[np.argsort(peaks[:, 0])] if len(peaks) else peaks

def spectral_cosine(peaks_a, peaks_b, tolerance=MZ_TOLERANCE_DA):
    a = np.asarray(peaks_a, dtype=float)
    b = np.asarray(peaks_b, dtype=float)
    if len(a) == 0 or len(b) == 0:
        return {'cosine': np.nan, 'matched_peak_n': 0, 'matched_a_fraction': np.nan, 'matched_b_fraction': np.nan}
    ia = np.sqrt(np.clip(a[:, 1], 0, None)); ib = np.sqrt(np.clip(b[:, 1], 0, None))
    ia = ia / np.linalg.norm(ia); ib = ib / np.linalg.norm(ib)
    candidates = []
    for i, mz_a in enumerate(a[:, 0]):
        left = np.searchsorted(b[:, 0], mz_a - tolerance, side='left')
        right = np.searchsorted(b[:, 0], mz_a + tolerance, side='right')
        for j in range(left, right):
            candidates.append((ia[i] * ib[j], i, j))
    used_a, used_b, score = set(), set(), 0.0
    for contribution, i, j in sorted(candidates, reverse=True):
        if i not in used_a and j not in used_b:
            used_a.add(i); used_b.add(j); score += contribution
    return {
        'cosine': float(score), 'matched_peak_n': len(used_a),
        'matched_a_fraction': len(used_a) / len(a), 'matched_b_fraction': len(used_b) / len(b),
    }

supported_adduct = {'positive': '[M+H]+', 'negative': '[M-H]-'}
measured_candidates = public_eligible.loc[
    public_eligible['inchikey_2d'].astype(str).isin(set(pilot['inchikey_2d'].astype(str)))
    & pd.to_numeric(public_eligible['collision_energy_value'], errors='coerce').notna()
].copy()
measured_candidates['mode'] = measured_candidates['ion_mode'].astype(str).str.lower()
measured_candidates = measured_candidates.loc[
    measured_candidates.apply(lambda row: str(row['adduct']) == supported_adduct.get(row['mode']), axis=1)
].copy()
measured_candidates['collision_energy_value'] = pd.to_numeric(measured_candidates['collision_energy_value'], errors='coerce')

pair_rows = []
if int(simulated_spectra['parse_success'].sum()) > 0:
    for sim in simulated_spectra.loc[simulated_spectra['parse_success']].itertuples(index=False):
        options = measured_candidates.loc[
            measured_candidates['inchikey_2d'].astype(str).eq(str(sim.inchikey_2d))
            & measured_candidates['mode'].eq(sim.mode)
        ].copy()
        if len(options) == 0:
            continue
        options['ce_delta'] = (options['collision_energy_value'] - sim.collision_energy_ev).abs()
        options = options.sort_values(['ce_delta', 'cleaned_peak_n', 'identity_ppm_error', 'spectrum_id'],
                                      ascending=[True, False, True, True])
        chosen = options.iloc[0]
        if float(chosen['ce_delta']) > MAX_CE_DIFFERENCE_EV:
            continue
        sim_peaks = parsed_peaks.loc[
            parsed_peaks['spectrum_task_id'].eq(sim.spectrum_task_id),
            ['fragment_mz', 'relative_intensity'],
        ].to_numpy(float)
        with PATHS['public_stream'].open('rb') as stream:
            stream.seek(int(chosen['stream_offset']))
            actual_key, record = pickle.load(stream)
        assert str(actual_key) == str(sim.inchikey_2d)
        assert str(record.get('source_library')) != 'ChemVista'
        observed_peaks = extract_observed_peaks(record)
        metrics = spectral_cosine(sim_peaks, observed_peaks)
        pair_rows.append({
            'spectrum_task_id': sim.spectrum_task_id, 'inchikey_2d': sim.inchikey_2d,
            'mode': sim.mode, 'adduct': sim.adduct, 'simulated_ce_ev': sim.collision_energy_ev,
            'measured_spectrum_id': chosen['spectrum_id'],
            'measured_ce_ev': chosen['collision_energy_value'], 'ce_delta_ev': chosen['ce_delta'],
            'simulated_peak_n': len(sim_peaks), 'measured_peak_n': len(observed_peaks),
            **metrics,
        })

sim_to_real = pd.DataFrame(pair_rows)
sim_to_real.to_csv(OUTPUT_DIR / 'cfm_sim_to_real_metrics.csv', index=False, encoding='utf-8-sig')
pair_coverage = len(sim_to_real) / max(int(simulated_spectra['parse_success'].sum()), 1)
print('Comparable sim-to-real pairs:', len(sim_to_real))
print('Pair coverage among parsed spectra:', f'{pair_coverage:.1%}')
if len(sim_to_real):
    display(sim_to_real.groupby(['mode', 'simulated_ce_ev'])['cosine'].agg(['count', 'median', 'mean']))

In [ ]:
if len(sim_to_real):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    sns.boxplot(data=sim_to_real, x='simulated_ce_ev', y='cosine', hue='mode', ax=axes[0])
    axes[0].set_title('CFM vs public measured spectra')
    axes[0].set_xlabel('CFM collision energy (eV)')
    axes[0].set_ylabel('Spectral cosine (0.02 Da)')
    sns.scatterplot(data=sim_to_real, x='simulated_peak_n', y='cosine', hue='mode', alpha=0.65, ax=axes[1])
    axes[1].set_title('Similarity versus simulated peak count')
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / '13_cfm_sim_to_real_diagnostics.png', dpi=240, bbox_inches='tight')
    plt.show()
else:
    print('尚无可绘制的 sim-to-real 配对；完整 Pilot 完成后重新运行此单元格。')

## 8. 成功率、失败模式与大规模资源估算

资源估算以成功任务的实测中位耗时/输出大小为基准，外推到当前全部 1,238 个 Development 化合物的正、负模式。
这是单进程保守估算；并行运行需要另行考虑 CPU、内存和 Docker I/O，不能简单按核心数线性缩短。

In [ ]:
completed_states = {'success', 'cached_complete'}
if len(execution_status):
    status_merged = execution_status.merge(
        jobs[['job_id', 'exact_mw', 'elements', 'mode']], on=['job_id', 'mode'], how='left', suffixes=('', '_job')
    )
    status_merged['completed'] = status_merged['status'].isin(completed_states)
    command_success_rate = float(status_merged['completed'].mean())
    mode_success = status_merged.groupby('mode')['completed'].agg(['count', 'sum', 'mean']).reset_index()
    good_runtime = pd.to_numeric(status_merged.loc[status_merged['completed'], 'runtime_seconds'], errors='coerce').dropna()
    good_bytes = pd.to_numeric(status_merged.loc[status_merged['completed'], 'output_bytes'], errors='coerce').dropna()
    median_seconds = float(good_runtime.median()) if len(good_runtime) else np.nan
    median_bytes = float(good_bytes.median()) if len(good_bytes) else np.nan
else:
    status_merged = pd.DataFrame()
    command_success_rate = np.nan
    mode_success = pd.DataFrame(columns=['mode', 'count', 'sum', 'mean'])
    median_seconds = np.nan
    median_bytes = np.nan

full_development_jobs = int(len(development) * 2)
estimate = pd.DataFrame([{
    'basis_completed_jobs': int(len(status_merged.loc[status_merged.get('completed', pd.Series(dtype=bool))])) if len(status_merged) else 0,
    'median_seconds_per_compound_mode': median_seconds,
    'median_output_bytes_per_compound_mode': median_bytes,
    'projected_development_compound_mode_jobs': full_development_jobs,
    'projected_single_process_hours': median_seconds * full_development_jobs / 3600 if np.isfinite(median_seconds) else np.nan,
    'projected_raw_output_gb': median_bytes * full_development_jobs / (1024 ** 3) if np.isfinite(median_bytes) else np.nan,
}])
estimate.to_csv(OUTPUT_DIR / 'cfm_resource_estimate.csv', index=False)
mode_success.to_csv(OUTPUT_DIR / 'cfm_mode_success_rates.csv', index=False)

if len(status_merged):
    failures = status_merged.loc[~status_merged['completed']].copy()
    failures.to_csv(OUTPUT_DIR / 'cfm_failed_jobs.csv', index=False, encoding='utf-8-sig')
    display(mode_success)
    display(estimate)
    if len(failures):
        display(failures[['job_id', 'mode', 'exact_mw', 'elements', 'status', 'stderr_tail']].head(20))
else:
    print('批量 Pilot 尚未执行，因此暂不生成正式资源估算。')

## 9. Pilot 门控、运行清单和 14 号决策

进入 14 号 Notebook 的建议门槛：

- 150 个 Pilot 化合物及 300 个正/负模式命令全部进入任务清单；
- 命令成功率至少 95%，且正、负模式分别至少 90%；
- 所有成功命令均可解析出 10/20/40 eV 三张非空谱；
- External Novel 重叠为 0，ChemVista 谱图读取为 0；
- Docker 固定标签及 digest 已记录；
- 至少形成一批严格同结构、同模式、同加合物、近碰撞能量的 sim-to-real 配对。

sim-to-real cosine 本身不设“一刀切”通过阈值；应根据模式、能量和结构分层分布决定模拟谱在后续训练中的权重，
而不是因均值不高就删除所有困难结构。

In [ ]:
n_commands_complete = int(status_merged['completed'].sum()) if len(status_merged) else 0
n_spectra_complete = int(simulated_spectra['parse_success'].sum())
positive_rate = float(mode_success.loc[mode_success['mode'].eq('positive'), 'mean'].iloc[0]) if (mode_success['mode'] == 'positive').any() else np.nan
negative_rate = float(mode_success.loc[mode_success['mode'].eq('negative'), 'mean'].iloc[0]) if (mode_success['mode'] == 'negative').any() else np.nan

qc_rows = [
    {'check': 'pilot_count_150', 'passed': len(pilot) == PILOT_N, 'observed': len(pilot), 'expected': PILOT_N},
    {'check': 'job_count_300', 'passed': len(jobs) == PILOT_N * 2, 'observed': len(jobs), 'expected': PILOT_N * 2},
    {'check': 'spectrum_task_count_900', 'passed': len(spectrum_tasks) == PILOT_N * 2 * 3, 'observed': len(spectrum_tasks), 'expected': PILOT_N * 2 * 3},
    {'check': 'external_novel_overlap_zero', 'passed': not bool(set(pilot['inchikey_2d']) & external_novel_keys), 'observed': len(set(pilot['inchikey_2d']) & external_novel_keys), 'expected': 0},
    {'check': 'fixed_image_digest_recorded', 'passed': CFM_IMAGE_DIGEST != 'unavailable', 'observed': CFM_IMAGE_DIGEST, 'expected': 'non-empty digest'},
    {'check': 'all_commands_completed', 'passed': n_commands_complete == len(jobs), 'observed': n_commands_complete, 'expected': len(jobs)},
    {'check': 'all_spectra_parsed', 'passed': n_spectra_complete == len(spectrum_tasks), 'observed': n_spectra_complete, 'expected': len(spectrum_tasks)},
    {'check': 'overall_command_success_ge_95pct', 'passed': bool(np.isfinite(command_success_rate) and command_success_rate >= 0.95), 'observed': command_success_rate, 'expected': '>=0.95'},
    {'check': 'positive_success_ge_90pct', 'passed': bool(np.isfinite(positive_rate) and positive_rate >= 0.90), 'observed': positive_rate, 'expected': '>=0.90'},
    {'check': 'negative_success_ge_90pct', 'passed': bool(np.isfinite(negative_rate) and negative_rate >= 0.90), 'observed': negative_rate, 'expected': '>=0.90'},
    {'check': 'sim_to_real_pairs_present', 'passed': len(sim_to_real) > 0, 'observed': len(sim_to_real), 'expected': '>0'},
]
qc = pd.DataFrame(qc_rows)
qc.to_csv(OUTPUT_DIR / 'cfm_pilot_qc_checks.csv', index=False, encoding='utf-8-sig')

completion_checks = ['all_commands_completed', 'all_spectra_parsed', 'overall_command_success_ge_95pct',
                     'positive_success_ge_90pct', 'negative_success_ge_90pct']
PILOT_STATUS = 'PASS_FOR_NOTEBOOK_14' if qc.loc[qc['check'].isin(completion_checks), 'passed'].all() else 'INCOMPLETE_OR_REQUIRES_REVIEW'
display(qc)
print('PILOT_STATUS:', PILOT_STATUS)

output_files = [p for p in OUTPUT_DIR.rglob('*') if p.is_file() and p.name != 'cfm_pilot_run_manifest.json']
run_manifest = {
    'notebook': '13_CFM小规模Pilot与运行条件确认.ipynb',
    'generated_at_utc': utc_now(),
    'status': PILOT_STATUS,
    'scientific_scope': 'Development-only CFM-ID pilot; ChemVista spectra and blind labels remain locked',
    'external_test_unlocked': False,
    'selection': {
        'seed': SEED, 'pilot_n': PILOT_N,
        'axes': ['molecular_weight_tertile', 'measured_mode_profile', 'element_class',
                 'primary_endpoint_availability', 'frozen_cv_fold'],
    },
    'cfm': {
        'docker_image': CFM_IMAGE, 'docker_image_digest': CFM_IMAGE_DIGEST,
        'probability_threshold': PROBABILITY_THRESHOLD,
        'adducts': {mode: value['adduct'] for mode, value in MODE_CONFIG.items()},
        'energy_mapping_ev': COLLISION_ENERGIES,
        'annotate_fragments': False,
    },
    'counts': {
        'development_compounds': len(development), 'pilot_compounds': len(pilot),
        'command_jobs_expected': len(jobs), 'command_jobs_complete': n_commands_complete,
        'simulated_spectra_expected': len(spectrum_tasks), 'simulated_spectra_parsed': n_spectra_complete,
        'sim_to_real_pairs': len(sim_to_real),
    },
    'sim_to_real_policy': {
        'source': 'public Development spectra only', 'adduct_exact_match': True,
        'max_ce_difference_ev': MAX_CE_DIFFERENCE_EV, 'mz_tolerance_da': MZ_TOLERANCE_DA,
        'metric': 'greedy cosine after square-root intensity transform',
    },
    'software': {
        'python': sys.version, 'platform': platform.platform(), 'rdkit': rdBase.rdkitVersion,
        'docker_client': docker_client.get('stdout', ''), 'docker_server': docker_server.get('stdout', ''),
    },
    'input_hashes': {
        name: sha256_file(path) for name, path in PATHS.items() if name != 'public_stream'
    },
    'public_stream_provenance': {
        'path': str(PATHS['public_stream']), 'size_bytes': PATHS['public_stream'].stat().st_size,
        'selected_peak_hashes_carried_by_public_manifest': True,
    },
    'output_hashes': {str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in output_files},
}
manifest_path = OUTPUT_DIR / 'cfm_pilot_run_manifest.json'
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Run manifest:', manifest_path)

## 10. 如何解释结果与下一步

- 如果 `PILOT_STATUS = PASS_FOR_NOTEBOOK_14`：进入 14 号，冻结大规模 CFM 候选结构和无泄漏分配规则。
- 如果只是 `all_commands_completed` 未通过：先查看 `cfm_failed_jobs.csv`，按错误类型处理后断点重跑，不要改变 Pilot 样本。
- 如果失败集中在超大分子、金属或特殊元素：把这一事实定义成 CFM 的适用域边界，并在 14 号中预先冻结排除/保留失败状态规则。
- 如果正负模式成功率差异明显：后续模拟谱必须保留 `mode`，不能将两种模式直接混成同一种输入。
- 如果 sim-to-real 分布随能量明显变化：15 号应设计能量匹配、随机能量采样或相似度加权消融。

**不要在 13 号中训练 Transformer，也不要解锁 ChemVista。** 13 号的任务是证明 CFM 生成链条可运行、可追踪、
可解析，并估计扩大规模的代价与风险。